# Regression Analysis on Cluster and Individual Features

## Overview
Transmission clusters were identified using EpiLink, which integrates pairwise genetic and temporal distances within 3-week sliding windows (1-week stride), applying Leiden community detection across resolutions 0.1–0.8. Cluster- and sequence-level summaries were then modelled using Bayesian binomial and count regression in Bambi to examine whether area-level deprivation (SIMD) predicts transmission cluster size and individual clustering consistency, adjusting for sequencing effort and key confounders.

---

## Methods

### Study Design
We conducted a multi-level Bayesian analysis of SARS-CoV-2 transmission dynamics in Scotland, integrating genomic surveillance data with area-level socioeconomic deprivation measures from the Scottish Index of Multiple Deprivation (SIMD). Analysis was performed at two complementary levels: transmission clusters and individual sequences.

---

### Genomic Data and Quality Control
Whole-genome sequences passing quality control (`nextclade_qc = "good"`) were included. Each sequence was linked to the patient's residential datazone, enabling linkage to SIMD quintile, decile, and domain-level deprivation ranks (income, employment, education, health, access, crime, and housing).

---

### Transmission Cluster Detection — EpiLink
Transmission clusters were identified using EpiLink, which integrates two sources of pairwise epidemiological evidence:

- **Consensus genetic distance** — the number of mutations separating any two sequences
- **Temporal distance** — the difference in sample collection dates between any two cases

Combining genetic and temporal proximity into a unified pairwise distance allows EpiLink to capture transmission linkage more robustly than approaches relying on genetic distance alone, which can conflate phylogenetically similar but temporally distant sequences.

Clustering was applied within a sliding window framework: each window spanned 3 weeks, advancing in 1-week strides, so any given sequence could appear in up to three consecutive windows. Within each window, Leiden community detection was applied across resolutions 0.1–0.8, where lower resolutions produce coarser partitions into larger clusters and higher resolutions produce finer, more fragmented ones. A cluster was defined as non-singleton if it contained more than one sequence. This multi-resolution, multi-window design yields robust cluster assignments that are not sensitive to any single resolution or time window.

---

### Data Aggregation

**Sequence-level.** For each unique sequence, observations were aggregated across all window × resolution combinations in which it appeared. The outcome was the clustering consistency rate: `non_singleton_k` (evaluations in which the sequence fell in a non-singleton cluster) out of `non_singleton_n` (total evaluations). High rates indicate robust clustering across temporal and resolution contexts; low rates indicate isolated or inconsistently linked cases.

**Cluster-level.** For each unique cluster observed across windows and resolutions, aggregate summaries were computed: size (`n_sequences`), modal SIMD quintile of member sequences (`simd_quintile_mode`), within-cluster deprivation heterogeneity (`simd_quintile_std`), age structure (`median_age`, `age_diversity`), sex composition (`frac_female`), and vaccination coverage (`frac_vaccinated`).

---

### Statistical Analysis
All models were fit using Bayesian inference in Bambi, with posterior distributions estimated via MCMC. Convergence was assessed using the Gelman–Rubin statistic (r̂ ≤ 1.01) and effective sample size (ESS > 400). SIMD domain predictors were standardised as negated z-scores so that higher values consistently indicate greater deprivation, enabling comparison across domains on a common scale.

#### Cluster-level models
Cluster size was modelled as `n_sequences − 1` (secondary sequences beyond the index case, anchoring singletons at zero) with a negative binomial likelihood. `log_seq_prop` was included as an offset to adjust for sequencing effort, and a window-level random effect `(1 | window_id)` accounted for non-independence of clusters observed within the same window.

#### Individual-level models
The clustering consistency rate (`non_singleton_k / non_singleton_n`) was modelled with a binomial likelihood. `log_seq_prop` (log geometric mean of window-level sequencing proportion) was included as a covariate to adjust for the fact that higher local sequencing effort mechanically increases the probability of detecting transmission links. SIMD quintile was coded with quintile 3 (middle deprivation) as the reference category.

In [1]:
from utils import data
from regression_models import  get_model_registry, run_models

In [2]:
cluster_data = data.load_cluster_features()
individual_data = data.load_individual_features()

In [3]:
display(get_model_registry().to_pandas())

,model,level,description,fixed_effects,interaction_effects
0,deprivation_main,cluster,Does area deprivation predict cluster size?,"[C(simd_quintile_mode, Treatment(3))]",[]
1,deprivation_x_epoch,cluster,Does the deprivation effect on cluster size va...,"[C(simd_quintile_mode, Treatment(3)), C(epoch)]","[C(simd_quintile_mode, Treatment(3)):C(epoch)]"
2,within_cluster_mixing,cluster,Does socioeconomic mixing within clusters pred...,"[C(simd_quintile_mode, Treatment(3)), simd_qui...",[]
3,vaccination_deprivation,cluster,Does vaccination coverage moderate cluster siz...,"[C(simd_quintile_mode, Treatment(3)), frac_vac...",[]
4,domain_income,cluster,Does the income deprivation domain drive clust...,"[income_zscore, median_age, age_diversity, fra...",[]
5,domain_employment,cluster,Does the employment deprivation domain drive c...,"[employment_zscore, median_age, age_diversity,...",[]
6,domain_education,cluster,Does the education deprivation domain drive cl...,"[education_zscore, median_age, age_diversity, ...",[]
7,domain_health,cluster,Does the health deprivation domain drive clust...,"[health_zscore, median_age, age_diversity, fra...",[]
8,domain_access,cluster,Does the access deprivation domain drive clust...,"[access_zscore, median_age, age_diversity, fra...",[]
9,domain_crime,cluster,Does the crime deprivation domain drive cluste...,"[crime_zscore, median_age, age_diversity, frac...",[]


In [ ]:
models = []

In [3]:
cluster_traces, individual_traces = reg.run_all(cluster_data, individual_data)

[deprivation_main] Loading cached trace from bambi_outputs/deprivation_main_binomial_trace.nc
[deprivation_main_epoch] Fitting  : proportion(non_singleton_k, non_singleton_n) ~ C(dz_simd_quintile, Treatment(3)) + log_seq_prop + C(dz_simd_quintile, Treatment(3)):C(epoch)


Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [Intercept, C(dz_simd_quintile, Treatment(3)), log_seq_prop, C(dz_simd_quintile, Treatment(3)):C(epoch)]


Output()

ValueError: Not enough samples to build a trace.